## Proceso estadistico de los datos
- En base a los CSV creados anteriormente, vamos a sacar estadisticas sobre los datos y su contenido.

In [1]:
## Importar dependencias
import glob
import os
from io import StringIO
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd

In [3]:
## Recuperacion de los archivos CSV

# Cargar variables del archivo .env
load_dotenv()

## Diccionario de PATHs
PATHS = {
    "Tarea2_CSV_train": Path(os.environ["path_metalriskES2023_task2_CSV_train"]),
    "Tarea2_CSV_test": Path(os.environ["path_metalriskES2023_task2_CSV_test"]),
    "Tarea2_CSV_trial": Path(os.environ["path_metalriskES2023_task2_CSV_trial"])
}

## Variables de rutas
# Tarea 2 rutas
ruta_guardado_tarea2_csv_train = PATHS["Tarea2_CSV_train"]
ruta_guardado_tarea2_csv_trial = PATHS["Tarea2_CSV_trial"]
ruta_guardado_tarea2_csv_test = PATHS["Tarea2_CSV_test"]

# Nombres de archivos 
nombre_archivo_tarea2_A_csv = "Tarea_A.csv"



In [34]:
## Configuraciones

direcciones = [ruta_guardado_tarea2_csv_train,
               ruta_guardado_tarea2_csv_test,
               ruta_guardado_tarea2_csv_trial]


for path in direcciones:

    ## Cargamos las etiquetas GOLD de la tarea 2 A
    ruta_tarea2_a_csv = os.path.join(path, 
                                     nombre_archivo_tarea2_A_csv)


    datos_CSV_tarea2_A_train = pd.read_csv(ruta_tarea2_a_csv)

    ## Estadistica en los datos de Tarea 2 A Train

    # Longitud en caracteres
    datos_CSV_tarea2_A_train["longitud_caracteres"] = datos_CSV_tarea2_A_train["mensaje"].fillna("").astype(str).str.len()

    # Longitud en palabras
    datos_CSV_tarea2_A_train["longitud_palabras"] = (datos_CSV_tarea2_A_train["mensaje"].fillna("").astype(str).str.split().str.len())


    print(f"========================== {nombre_archivo_tarea2_A_csv} =====================================")
    print(f"========================== { path } =============================")
    print("==================================================================")
    # Resumen estadístico de ambas métricas
    print(datos_CSV_tarea2_A_train[["longitud_caracteres", "longitud_palabras"]].describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]))

    # Calidad de los datos
    print("Registros:", len(datos_CSV_tarea2_A_train))
    print("Nulos:", datos_CSV_tarea2_A_train["mensaje"].isna().sum())
    print("Vacíos:", datos_CSV_tarea2_A_train["mensaje"].fillna("").str.strip().eq("").sum())
    print("Duplicados exactos:", datos_CSV_tarea2_A_train["mensaje"].dropna().duplicated().sum())

    # Vocabulario y palabras frecuentes
    palabras = (datos_CSV_tarea2_A_train["mensaje"].dropna().astype(str).str.lower().str.findall(r"\b\w+\b").explode())

    print("Palabras totales:", len(palabras))
    print("Vocabulario único:", palabras.nunique())
    print("Top 20 palabras:")
    print(palabras.value_counts().head(5))
    print("===================================================================")


========================== Tarea_A.csv =====================================
========================== /home/octavio/Trabajo_tesis/Datos/corpusMetalRiskESCSV/MetalRiskES2023/Tarea_2/Train =============================
       longitud_caracteres  longitud_palabras
count          6248.000000        6248.000000
mean             69.887644          14.119558
std             127.704173          24.759783
min               7.000000           1.000000
25%              28.000000           6.000000
50%              43.000000           9.000000
75%              75.000000          15.000000
90%             130.000000          26.000000
95%             187.650000          37.000000
max            4279.000000         783.000000
Registros: 6248
Nulos: 0
Vacíos: 0
Duplicados exactos: 0
Palabras totales: 81287
Vocabulario único: 10309
Top 20 palabras:
mensaje
que    3274
de     2659
y      2272
no     2133
a      1582
Name: count, dtype: int64
========================== Tarea_A.csv ===================

In [9]:
ARCHIVOS = {
    "Train": PATHS["Tarea2_CSV_train"] / nombre_archivo_tarea2_A_csv,
    "Trial": PATHS["Tarea2_CSV_trial"] / nombre_archivo_tarea2_A_csv,
    "Test":  PATHS["Tarea2_CSV_test"] / nombre_archivo_tarea2_A_csv,
}

datos = {}

for nombre, ruta in ARCHIVOS.items():
    df = pd.read_csv(ruta)
    datos[nombre] = df

    print(f"\n{'=' * 50}")
    print(f"{nombre}: {len(df):,} mensajes")

    print("\nDistribución de etiquetas:")
    print(df["etiqueta_gold"].value_counts().sort_index())
    print("\nPorcentaje por etiqueta:")
    print(
        (df["etiqueta_gold"].value_counts(normalize=True)
         .sort_index() * 100).round(2)
    )

    print(f"\nSujetos únicos: {df['sujeto'].nunique():,}")
    print("Etiquetas distintas:", sorted(df["etiqueta_gold"].unique()))

    # Verifica que cada sujeto tenga una etiqueta consistente.
    etiquetas_por_sujeto = df.groupby("sujeto")["etiqueta_gold"].nunique()
    inconsistentes = (etiquetas_por_sujeto > 1).sum()
    print("Sujetos con etiquetas inconsistentes:", inconsistentes)

# Comprueba si hay participantes compartidos entre conjuntos.
print(f"\n{'=' * 50}")
print("SOLAPAMIENTO DE SUJETOS")

for a, b in [("Train", "Trial"), ("Train", "Test"), ("Trial", "Test")]:
    sujetos_a = set(datos[a]["sujeto"].dropna().unique())
    sujetos_b = set(datos[b]["sujeto"].dropna().unique())
    comunes = sujetos_a & sujetos_b

    print(f"{a} - {b}: {len(comunes)} sujetos compartidos")

# Revisa valores inesperados.
for nombre, df in datos.items():
    etiquetas = set(df["etiqueta_gold"].dropna().unique())
    print(f"{nombre}: etiquetas fuera de 0/1 = {etiquetas - {0, 1}}")



Train: 6,248 mensajes

Distribución de etiquetas:
etiqueta_gold
0    3135
1    3113
Name: count, dtype: int64

Porcentaje por etiqueta:
etiqueta_gold
0    50.18
1    49.82
Name: proportion, dtype: float64

Sujetos únicos: 175
Etiquetas distintas: [np.int64(0), np.int64(1)]
Sujetos con etiquetas inconsistentes: 0

Trial: 624 mensajes

Distribución de etiquetas:
etiqueta_gold
0    327
1    297
Name: count, dtype: int64

Porcentaje por etiqueta:
etiqueta_gold
0    52.4
1    47.6
Name: proportion, dtype: float64

Sujetos únicos: 10
Etiquetas distintas: [np.int64(0), np.int64(1)]
Sujetos con etiquetas inconsistentes: 0

Test: 5,164 mensajes

Distribución de etiquetas:
etiqueta_gold
0    2825
1    2339
Name: count, dtype: int64

Porcentaje por etiqueta:
etiqueta_gold
0    54.71
1    45.29
Name: proportion, dtype: float64

Sujetos únicos: 149
Etiquetas distintas: [np.int64(0), np.int64(1)]
Sujetos con etiquetas inconsistentes: 0

SOLAPAMIENTO DE SUJETOS
Train - Trial: 0 sujetos compartidos
T